In [4]:
import numpy as np
import pandas as pd

1. Загрузка данных

In [5]:
df = pd.read_csv('content/house_price_regression_dataset.csv')

https://www.kaggle.com/datasets/prokshitha/home-value-insights

In [5]:
# 'Square_Footage': Размер дома в квадратных футах. Более крупные дома,
# как правило, стоят дороже.

# 'Num_Bedrooms': Количество спален в доме. Большее количество спален
# обычно увеличивает стоимость дома.

# 'Num_Bathrooms': Количество ванных комнат в доме. Дома с большим количеством
# ванных комнат, как правило, стоят дороже.

# 'Year_Built': Год постройки дома. Более старые дома могут стоить дешевле
# из-за износа.

# 'Lot_Size': Размер участка, на котором построен дом, измеряется в акрах.
# Большие участки, как правило, увеличивают стоимость недвижимости.

# 'Garage_Size': Количество автомобилей, которые могут поместиться в гараже.
# Дома с большими гаражами обычно стоят дороже.

# 'Neighborhood_Quality': Оценка качества района по шкале от 1 до 10, где
# 10 означает высокое качество района. В лучших районах обычно более высокие цены.

# 'House_Price' (целевая переменная): цена дома, которая является зависимой
# переменной, которую вы пытаетесь предсказать.

In [6]:
df

,Square_Footage,Num_Bedrooms,Num_Bathrooms,Year_Built,Lot_Size,Garage_Size,Neighborhood_Quality,House_Price
0,1360,2,1,1981,0.599637,0,5,2.623829e+05
1,4272,3,3,2016,4.753014,1,6,9.852609e+05
2,3592,1,2,2016,3.634823,0,9,7.779774e+05
3,966,1,2,1977,2.730667,1,8,2.296989e+05
4,4926,2,1,1993,4.699073,0,8,1.041741e+06
...,...,...,...,...,...,...,...,...
995,3261,4,1,1978,2.165110,2,10,7.014940e+05
996,3179,1,2,1999,2.977123,1,10,6.837232e+05
997,2606,4,2,1962,4.055067,0,2,5.720240e+05
998,4723,5,2,1950,1.930921,0,7,9.648653e+05


2. Предобработка данных

In [7]:
# Проверим есть ли столбы с пропущенными значениями
df.isna().any()

Square_Footage          False
Num_Bedrooms            False
Num_Bathrooms           False
Year_Built              False
Lot_Size                False
Garage_Size             False
Neighborhood_Quality    False
House_Price             False
dtype: bool

3. Загрузка модели, обучение

https://scikit-learn.org/stable/modules/generated/sklearn.linear_model.LinearRegression.html

In [26]:
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split

model = LinearRegression()

X = df.drop(columns=['House_Price', 'Neighborhood_Quality'])
y = df['House_Price']



# Разбиваем на обучающую и тестовую выборки
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Обучаем
model.fit(X_train, y_train)

# Делаем предсказание
y_pred = model.predict(X_test)

4. Оценка модели

In [27]:
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error

# MSE (Среднеквадратичная ошибка): чем ближе к 0, тем лучше
mse = mean_squared_error(y_test, y_pred)

# RMSE (Корень из MSE): ошибка в тех же единицах, что и целевая переменная
# (например, в рублях или метрах)
rmse = mean_squared_error(y_test, y_pred, squared=False)

# MAE (Средняя абсолютная ошибка): среднее отклонение
mae = mean_absolute_error(y_test, y_pred)

# R^2 (Коэффициент детерминации): точность от 0 до 1
r2 = r2_score(y_test, y_pred)

print(mse)
print(rmse)
print(mae)
print(r2)

101211537.28459945
10060.394489511804
8164.595362522351
0.9984298273059786


/usr/lib/python3/dist-packages/sklearn/metrics/_regression.py:483: FutureWarning: 'squared' is deprecated in version 1.4 and will be removed in 1.6. To calculate the root mean squared error, use the function'root_mean_squared_error'.
  warnings.warn(


5. Эксперименты со скейлерами

Сравним несколько популярных способов масштабирования признаков и посмотрим, как они влияют на метрики регрессии.

In [ ]:
from sklearn.preprocessing import StandardScaler, MinMaxScaler, RobustScaler, PowerTransformer, QuantileTransformer
from sklearn.pipeline import make_pipeline

# Определяем набор скейлеров для сравнения
scalers = {
    'None': None,
    'Standard': StandardScaler(),
    'MinMax': MinMaxScaler(),
    'Robust': RobustScaler(),
    'Power': PowerTransformer(),
    'Quantile': QuantileTransformer(output_distribution='normal')
}

results = []
for name, scaler in scalers.items():
    if scaler is None:
        clf = LinearRegression()
    else:
        clf = make_pipeline(scaler, LinearRegression())
    # Обучаем и оцениваем
    clf.fit(X_train, y_train)
    y_pred_s = clf.predict(X_test)
    mse_s = mean_squared_error(y_test, y_pred_s)
    rmse_s = mean_squared_error(y_test, y_pred_s, squared=False)
    mae_s = mean_absolute_error(y_test, y_pred_s)
    r2_s = r2_score(y_test, y_pred_s)
    results.append({
        'scaler': name,
        'mse': mse_s,
        'rmse': rmse_s,
        'mae': mae_s,
        'r2': r2_s
    })

res_df = pd.DataFrame(results).set_index('scaler')
# Отображаем аккуратно округлённые значения
res_df.round(4)